## Audit direction aware edges

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # assumes notebook is in notebooks/, project root is one level up
sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence


In [2]:
from evaluation.benchmark import generate_questions
import pandas as pd

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")
questions = generate_questions(df, n_samples=15, seed=42)
for q in questions:
    print(f"query:  {q.query!r}")
    print(f"answer: {q.correct_answer!r}  (relation: {q.source_relation})\n")

c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


query:  'Who is Doramin a spouse of?'
answer: 'his little motherly witch of a wife'  (relation: spouse_of)

query:  'Who is Adair a protector of?'
answer: 'assistant engineer'  (relation: protector_of)

query:  'Who is Aurelius an enemy of?'
answer: 'vortigern'  (relation: enemy_of)

query:  'Who is The school an enemy of?'
answer: 'the enemy'  (relation: enemy_of)

query:  'Who is Mrs. Hare the mother of?'
answer: 'richard'  (relation: parent_mother_of)

query:  'Who is my wife a spouse of?'
answer: 'i'  (relation: spouse_of)

query:  'Who is Kali a leader of?'
answer: 'wahimas'  (relation: leader_of)

query:  'Who is Mr Brooke a companion of?'
answer: 'us'  (relation: companion_of)

query:  'Who is Pringle a member of?'
answer: 'school'  (relation: member_of)

query:  'Who is Death Valley located in?'
answer: 'armagosa range'  (relation: located_in)

query:  'Who is Mrs. Mirvan a companion of?'
answer: 'lord orville'  (relation: companion_of)

query:  'Who is M.C.C. a rival of?'
answ

In [3]:
# single-hop fix: does every generated question read
# unambiguously in the SAME direction as its own ground truth?
from evaluation.benchmark import generate_questions
from embedding.relation_text import relation_to_question, MANUAL_TEMPLATES

questions = generate_questions(df, n_samples=25, seed=42)

print(f"Generated {len(questions)} questions (candidate pool restricted to "
      f"{len(MANUAL_TEMPLATES)}/48 templated relation types)\n")
for q in questions[:25]:
    print(f"query:  {q.query!r}")
    print(f"answer: {q.correct_answer!r}  (relation: {q.source_relation})\n")

# Confirms scope-narrowing worked: no question should exist for a
# relation type without a verified template.
untemplated = [q for q in questions if relation_to_question("x", q.source_relation) is None]
print(f"Questions generated for untemplated relations (should be 0): {len(untemplated)}")

Generated 25 questions (candidate pool restricted to 31/48 templated relation types)

query:  'Who is Doramin a spouse of?'
answer: 'his little motherly witch of a wife'  (relation: spouse_of)

query:  'Who is Adair a protector of?'
answer: 'assistant engineer'  (relation: protector_of)

query:  'Who is Aurelius an enemy of?'
answer: 'vortigern'  (relation: enemy_of)

query:  'Who is The school an enemy of?'
answer: 'the enemy'  (relation: enemy_of)

query:  'Who is Mrs. Hare the mother of?'
answer: 'richard'  (relation: parent_mother_of)

query:  'Who is my wife a spouse of?'
answer: 'i'  (relation: spouse_of)

query:  'Who is Kali a leader of?'
answer: 'wahimas'  (relation: leader_of)

query:  'Who is Mr Brooke a companion of?'
answer: 'us'  (relation: companion_of)

query:  'Who is Pringle a member of?'
answer: 'school'  (relation: member_of)

query:  'Who is Death Valley located in?'
answer: 'armagosa range'  (relation: located_in)

query:  'Who is Mrs. Mirvan a companion of?'
answ

In [4]:
# regenerate the SAME three previously-confirmed-reversed
# cases (child_of/Will, protector_of/Taug, leader_of/King Arthur) and
# check the fix directly against the known-bad examples, not just new
# random samples.
from embedding.relation_text import relation_to_question

known_reversals = [
    ("Will", "child_of", "mrs. brand"),
    ("Taug", "protector_of", "teeka"),
    ("King Arthur", "leader_of", "dacia"),
]
for entity1, rel, correct_answer in known_reversals:
    q = relation_to_question(entity1, rel)
    print(f"{q!r}")
    print(f"  ground truth (entity2): {correct_answer!r}")
    print(f"  does the question's own English now point at entity2's role? (manual check)\n")

'Who is Will a child of?'
  ground truth (entity2): 'mrs. brand'
  does the question's own English now point at entity2's role? (manual check)

'Who is Taug a protector of?'
  ground truth (entity2): 'teeka'
  does the question's own English now point at entity2's role? (manual check)

'Who is King Arthur a leader of?'
  ground truth (entity2): 'dacia'
  does the question's own English now point at entity2's role? (manual check)



In [5]:
# n-hop chain fix: verify against real sampled paths, not
# just hand-picked examples, and confirm invalidated chains are being
# dropped rather than silently produced with a None phrase.
from evaluation.benchmark import find_n_hop_paths, _chain_phrase
import random
from graph.corpus import load_corpus


corpus = load_corpus("../data/graphs/corpus.pkl")

sample_book_id = "106"
graph = corpus[sample_book_id]

paths = find_n_hop_paths(graph, hops=2, max_samples=15)
valid, dropped = 0, 0
for p in paths:
    phrase = _chain_phrase(p["start"], p["relations"], p["directions"])
    if phrase is None:
        dropped += 1
        continue
    valid += 1
    print(f"{phrase!r}")
    print(f"  expected final answer: {p['end']!r}\n")

print(f"\n{valid} valid chains, {dropped} dropped (unverified relation in chain)")

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'younger apes'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a friend of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a companion of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that entity a friend of?'
  expected final answer: 'tantor'

'Who is taug a companion of? Then, who is that

In [6]:
from evaluation.benchmark import find_n_hop_paths, _chain_phrase

graph = corpus["106"]
paths = find_n_hop_paths(graph, hops=2, max_samples=30)

forward_only = [p for p in paths if all(d == "forward" for d in p["directions"])]
has_reverse = [p for p in paths if any(d == "reverse" for d in p["directions"])]
print(f"{len(forward_only)} all-forward, {len(has_reverse)} contain a reverse hop\n")

for p in has_reverse[:10]:
    phrase = _chain_phrase(p["start"], p["relations"], p["directions"])
    print(f"dirs={p['directions']} -> {phrase!r}")
    print(f"  expected: {p['end']!r}\n")

463 all-forward, 84 contain a reverse hop

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a friend of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a companion of that entity?'
  expected: 'tantor'

dirs=['forward', 'reverse'] -> 'Who is taug a companion of? Then, who is a friend of that en

In [7]:
import random
import pandas as pd

def graph_n_hop_search_OLD(graph, entity_a, hops):
    frontier = {entity_a}
    visited = {entity_a}
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            for u, v, _ in graph.edges(nbunch=[node], data=True):
                other = v if u == node else u
                if other not in visited:
                    next_frontier.add(other)
        visited |= next_frontier
        frontier = next_frontier
    return visited - {entity_a}

from evaluation.benchmark import graph_n_hop_search as graph_n_hop_search_NEW  # adjust to actual current location

rng = random.Random(42)
graph = corpus["106"]
sample_nodes = rng.sample(list(graph.nodes), min(30, graph.number_of_nodes()))

rows = []
for node in sample_nodes:
    for hops in (1, 2, 3):
        old_set = graph_n_hop_search_OLD(graph, node, hops)
        new_set = graph_n_hop_search_NEW(graph, node, hops)
        rows.append({"entity": node, "hops": hops,
                      "old_reachable": len(old_set), "new_reachable": len(new_set),
                      "recovered": len(new_set - old_set)})

df = pd.DataFrame(rows)
print(df.groupby("hops")[["old_reachable", "new_reachable", "recovered"]].mean())
print(f"\nentity/hop pairs with any newly-recovered nodes: {(df['recovered'] > 0).sum()} / {len(df)}")

      old_reachable  new_reachable  recovered
hops                                         
1          1.333333       2.066667   0.733333
2         18.066667      50.266667  32.200000
3         44.500000     114.966667  70.466667

entity/hop pairs with any newly-recovered nodes: 52 / 90


In [8]:
degrees = dict(graph.degree())
recovered_degrees, baseline_degrees = [], []

for node in sample_nodes:
    for hops in (1, 2, 3):
        old_set = graph_n_hop_search_OLD(graph, node, hops)
        new_set = graph_n_hop_search_NEW(graph, node, hops)
        newly_recovered = new_set - old_set
        recovered_degrees += [degrees[n] for n in newly_recovered]

import statistics
corpus_median_degree = statistics.median(degrees.values())
print(f"corpus-wide median degree: {corpus_median_degree}")
print(f"median degree of newly-recovered nodes: {statistics.median(recovered_degrees) if recovered_degrees else 'n/a'}")
print(f"mean degree of newly-recovered nodes: {statistics.mean(recovered_degrees) if recovered_degrees else 'n/a'} vs corpus mean: {statistics.mean(degrees.values()):.1f}")

corpus-wide median degree: 1
median degree of newly-recovered nodes: 2.0
mean degree of newly-recovered nodes: 17.054480980012894 vs corpus mean: 9.7


In [9]:
# Only reverse-traverse for relation types README confirmed are
# actually stored inconsistently in direction - everything else keeps
# the original forward-only semantics, matching what the evidence
# supports rather than the broadest possible interpretation of it.
SYMMETRIC_RELATIONS = {
    "companion_of", "friend_of", "enemy_of", "rival_of",
    "sibling_of", "spouse_of", "relative_of",
}

def graph_n_hop_search_SCOPED(graph, entity_a, hops):
    frontier = {entity_a}
    visited = {entity_a}
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            neighbors = {v for _, v, _ in graph.edges(nbunch=[node], data=True)}
            neighbors |= {u for u, _, d in graph.in_edges(nbunch=[node], data=True)
                          if d["relation"] in SYMMETRIC_RELATIONS}
            next_frontier |= neighbors - visited
        visited |= next_frontier
        frontier = next_frontier
    return visited - {entity_a}

# Three-way comparison, same sample_nodes as before - old (forward-only),
# new (all-direction), scoped (symmetric-only reverse).
import statistics
degrees = dict(graph.degree())
corpus_mean, corpus_median = statistics.mean(degrees.values()), statistics.median(degrees.values())

rows = []
for node in sample_nodes:
    for hops in (1, 2, 3):
        old_set = graph_n_hop_search_OLD(graph, node, hops)
        new_set = graph_n_hop_search_NEW(graph, node, hops)
        scoped_set = graph_n_hop_search_SCOPED(graph, node, hops)
        rows.append({
            "hops": hops, "old": len(old_set), "new": len(new_set), "scoped": len(scoped_set),
            "scoped_recovered": len(scoped_set - old_set),
        })

df = pd.DataFrame(rows)
print(df.groupby("hops")[["old", "new", "scoped", "scoped_recovered"]].mean())

scoped_recovered_degrees = []
for node in sample_nodes:
    for hops in (1, 2, 3):
        newly = graph_n_hop_search_SCOPED(graph, node, hops) - graph_n_hop_search_OLD(graph, node, hops)
        scoped_recovered_degrees += [degrees[n] for n in newly]

print(f"\nscoped version - median recovered degree: {statistics.median(scoped_recovered_degrees) if scoped_recovered_degrees else 'n/a'} (corpus median: {corpus_median})")
print(f"scoped version - mean recovered degree: {statistics.mean(scoped_recovered_degrees):.1f} (corpus mean: {corpus_mean:.1f})" if scoped_recovered_degrees else "n/a")

            old         new      scoped  scoped_recovered
hops                                                     
1      1.333333    2.066667    2.066667          0.733333
2     18.066667   50.266667   50.266667         32.200000
3     44.500000  114.966667  114.966667         70.466667

scoped version - median recovered degree: 2.0 (corpus median: 1)
scoped version - mean recovered degree: 17.1 (corpus mean: 9.7)


In [10]:
from evaluation.benchmark import generate_questions, evaluate_direction_aware
from embedding.vector_store import get_collection
from embedding.embed_relations import load_embedding_model

collection = get_collection()
model = load_embedding_model()
df = pd.read_parquet("../data/arf_chunks_parsed.parquet")

questions = generate_questions(df, n_samples=200, seed=42)
print(f"{len(questions)} questions generated (from 31/48 templated relation types)")

result = evaluate_direction_aware(collection, model, corpus, questions, k=5)
print(result)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2594.07it/s]


200 questions generated (from 31/48 templated relation types)
{'n_questions': 200, 'baseline_direction_correct': 0.315, 'hybrid_direction_correct': 0.585}


In [11]:
from evaluation.benchmark import generate_nhop_questions, evaluate_nhop_baseline, validate_graph_reachability

for hops in (1, 2, 3):
    requested = 100
    questions = generate_nhop_questions(corpus, hops=hops, n_samples=requested, seed=42)
    print(f"hops={hops}: requested={requested}, valid={len(questions)} "
          f"(dropped = untemplated relation somewhere in the chain)")

    baseline_acc = evaluate_nhop_baseline(collection, model, questions, k=10)
    reachability = validate_graph_reachability(corpus, questions)
    print(f"  baseline_acc={baseline_acc:.3f}")
    print(f"  reachability={reachability}\n")

hops=1: requested=100, valid=97 (dropped = untemplated relation somewhere in the chain)
  baseline_acc=0.392
  reachability={'n_questions': 97, 'agreement_with_independent_check': 1.0, 'reachable_rate': 1.0}

hops=2: requested=100, valid=91 (dropped = untemplated relation somewhere in the chain)
  baseline_acc=0.264
  reachability={'n_questions': 91, 'agreement_with_independent_check': 1.0, 'reachable_rate': 1.0}

hops=3: requested=100, valid=91 (dropped = untemplated relation somewhere in the chain)
  baseline_acc=0.231
  reachability={'n_questions': 91, 'agreement_with_independent_check': 1.0, 'reachable_rate': 1.0}

